In [ ]:
# ============================================================
# Cell 1 — Load EmotiBit EDA (Ice Test / Grant_Data)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter, lfilter_zi

# --- Paths ---
import os
DATA_DIR = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "Grant_Data"
)
# Use absolute path for robustness
DATA_DIR = (
    r"C:\Users\gloriosog\OneDrive - Milwaukee School of Engineering"
    r"\Year 4 Courses\Semester 1\Senior Design\Senior Design Repo"
    r"\stress-detection-wearable\Ice_Test\Grant_Data"
)
EA_FILE = os.path.join(DATA_DIR, "2026-04-21_11-32-56-286651_EA.csv")
TL_FILE = os.path.join(DATA_DIR, "2026-04-21_11-32-56-286651_TL.csv")

# --- Load EDA ---
# EmotiBit EA.csv columns:
#   LocalTimestamp, EmotiBitTimestamp, PacketNumber, DataLength,
#   TypeTag, ProtocolVersion, DataReliability, EA
df_ea = pd.read_csv(EA_FILE)
print("EA columns:", df_ea.columns.tolist())
print(df_ea.head(3))

# LocalTimestamp is Unix epoch in SECONDS (magnitude ~1.77e9)
timestamps_s = df_ea["LocalTimestamp"].values       # seconds since epoch
eda_raw      = df_ea["EA"].values.astype(float)     # microsiemens (μS)

# Relative time from first sample
t_rel = timestamps_s - timestamps_s[0]              # seconds from recording start

# Compute actual sample rate from the data
fs_eda = len(t_rel) / (t_rel[-1] - t_rel[0])
print(f"\nRecording duration : {t_rel[-1]:.1f} s  ({t_rel[-1]/60:.2f} min)")
print(f"Total EDA samples  : {len(eda_raw)}")
print(f"Estimated fs_eda   : {fs_eda:.2f} Hz  (nominal 15 Hz)")

# --- Protocol phase boundary variables (seconds from recording start) ---
t_cal_end    = 60    # Calibration ends at 60 s
t_base_end   = 315   # Baseline ends at 5 min 15 s
t_stress_end = 435   # Stress (CPT) ends at 7 min 15 s
# Recovery: t_stress_end → end of recording

print(f"\nPhase boundaries: cal={t_cal_end}s, baseline={t_base_end}s, stress={t_stress_end}s")

# --- Load TL (timestamp map) for reference ---
df_tl = pd.read_csv(TL_FILE)
print("\nTL columns:", df_tl.columns.tolist())
print(df_tl.head(3))


In [ ]:
# ============================================================
# Cell 2 — Raw EDA: Full Recording with Phase Labels
# ============================================================

fig, ax = plt.subplots(figsize=(15, 5))

# Raw EDA signal
ax.plot(t_rel, eda_raw, color='black', linewidth=0.8, label='Raw EDA')

# Phase shading with axvspan (alpha=0.15 per spec)
phase_spans = [
    (0,            t_cal_end,    'gray',  'Calibration'),
    (t_cal_end,    t_base_end,   'blue',  'Baseline'),
    (t_base_end,   t_stress_end, 'red',   'Stress (CPT)'),
    (t_stress_end, t_rel[-1],    'green', 'Recovery'),
]
for x0, x1, color, label in phase_spans:
    ax.axvspan(x0, x1, color=color, alpha=0.15, label=label)

# Vertical dashed lines at each boundary
for xv, annotation in [
    (t_cal_end,    f'Cal end\n({t_cal_end}s)'),
    (t_base_end,   f'Baseline end\n({t_base_end}s)'),
    (t_stress_end, f'Stress end\n({t_stress_end}s)'),
]:
    ax.axvline(xv, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.text(xv + 2, ax.get_ylim()[1] if ax.get_ylim()[1] != 1.0 else eda_raw.max(),
            annotation, fontsize=8, va='top', color='black')

ax.set_xlim(0, t_rel[-1])
ax.set_xlabel('Time from recording start (s)', fontsize=12)
ax.set_ylabel('EDA (μS)', fontsize=12)
ax.set_title('Raw EDA — Full Recording with Phase Labels', fontsize=14)
ax.legend(loc='upper right', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 3 — Causal EDA Decomposition (Full Recording) + 3-Panel Plot
# ============================================================
from matplotlib.patches import Patch

def extract_causal_eda(raw_eda, fs):
    nyq = 0.5 * fs
    b1, a1 = butter(1, 1.0 / nyq, btype='low')
    zi1 = lfilter_zi(b1, a1) * raw_eda[0]
    smoothed_eda, _ = lfilter(b1, a1, raw_eda, zi=zi1)
    b2, a2 = butter(2, 0.05 / nyq, btype='low')
    zi2 = lfilter_zi(b2, a2) * smoothed_eda[0]
    tonic_causal, _ = lfilter(b2, a2, smoothed_eda, zi=zi2)
    dt = 1.0 / fs
    derivative = np.gradient(smoothed_eda, dt)
    phasic_causal = np.maximum(0, derivative)
    return {'smoothed': smoothed_eda, 'tonic': tonic_causal, 'phasic': phasic_causal}

# Apply to the FULL recording
print("Running causal EDA decomposition on full recording...")
components = extract_causal_eda(eda_raw, fs_eda)
smoothed = components['smoothed']
tonic    = components['tonic']
phasic   = components['phasic']
print("Decomposition complete. Plotting...")

# Reusable phase span definition
phase_spans = [
    (0,            t_cal_end,    'gray',  'Calibration'),
    (t_cal_end,    t_base_end,   'blue',  'Baseline'),
    (t_base_end,   t_stress_end, 'red',   'Stress (CPT)'),
    (t_stress_end, t_rel[-1],    'green', 'Recovery'),
]

# 3-subplot figure, shared x-axis
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

subplot_cfg = [
    (axes[0], smoothed, 'black',  'Smoothed EDA (1 Hz causal lowpass)',    'EDA (μS)'),
    (axes[1], tonic,    'purple', 'Tonic Component (0.05 Hz causal SCL)',   'Tonic (μS)'),
    (axes[2], phasic,   'green',  'Phasic Component (rate-of-change > 0)',  'dEDA/dt (μS/s)'),
]

for ax, sig, color, title, ylabel in subplot_cfg:
    ax.plot(t_rel, sig, color=color, linewidth=0.8)

    # Phase shading on every subplot
    for x0, x1, span_color, _ in phase_spans:
        ax.axvspan(x0, x1, color=span_color, alpha=0.15)

    # Phase boundary vertical lines
    for xv in [t_cal_end, t_base_end, t_stress_end]:
        ax.axvline(xv, color='black', linestyle='--', linewidth=1.0, alpha=0.6)

    ax.set_title(title, fontsize=13)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, t_rel[-1])

# Phase legend on the top subplot only
legend_patches = [
    Patch(facecolor='gray',  alpha=0.3, label='Calibration'),
    Patch(facecolor='blue',  alpha=0.3, label='Baseline'),
    Patch(facecolor='red',   alpha=0.3, label='Stress (CPT)'),
    Patch(facecolor='green', alpha=0.3, label='Recovery'),
]
axes[0].legend(handles=legend_patches, loc='upper right', fontsize=9)

axes[2].set_xlabel('Time from recording start (s)', fontsize=12)
fig.suptitle('EmotiBit EDA — Causal Decomposition, Full Recording', fontsize=15)
plt.tight_layout()
plt.show()
